# WS_03b · Khảo sát nhãn yếu

Notebook này **chỉ đọc** kết quả `weak_labels/v1` và mệnh đề WS_02. Notebook không gọi NLI, ConceptCLIP hoặc LLM, và không thay đổi `labels.npz`.

Các bảng, hình và `survey_summary.md` được ghi vào `weak_labels/v1/survey/`.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

WEAK_LABELS_DIR = Path('/content/drive/MyDrive/Cao học/WeakSup/weak_labels/v1')
PROPOSITIONS_PATH = Path('/content/drive/MyDrive/Cao học/WeakSup/propositions/v1/propositions_final.json')
LABELS_PATH = WEAK_LABELS_DIR / 'labels.npz'
META_PATH = WEAK_LABELS_DIR / 'meta.json'
SURVEY_DIR = WEAK_LABELS_DIR / 'survey'

for required_path in (LABELS_PATH, META_PATH, PROPOSITIONS_PATH):
    if not required_path.is_file():
        raise FileNotFoundError(f'Missing required read-only input: {required_path}')

with np.load(LABELS_PATH, allow_pickle=False) as archive:
    arrays = {name: archive[name] for name in archive.files}
with META_PATH.open(encoding='utf-8') as handle:
    meta = json.load(handle)
with PROPOSITIONS_PATH.open(encoding='utf-8') as handle:
    propositions = pd.DataFrame(json.load(handle))

required_arrays = {
    'case_id', 'split', 'proposition_id', 'label_text_nli', 'label_text',
    'llm_group', 'llm_quote_invalid', 'label_img', 'label_final',
    'p_entail', 'p_contra', 'p_img',
}
missing = required_arrays - arrays.keys()
if missing:
    raise ValueError(f'labels.npz lacks required arrays: {sorted(missing)}')

label_ids = arrays['proposition_id'].astype(str)
proposition_ids = propositions['proposition_id'].astype(str).to_numpy()
if not np.array_equal(label_ids, proposition_ids):
    raise ValueError('proposition_id order in labels.npz differs from propositions_final.json.')
if len(arrays['case_id']) != arrays['label_final'].shape[0] or len(label_ids) != arrays['label_final'].shape[1]:
    raise ValueError('labels.npz case/proposition dimensions are inconsistent.')

required_categories = {'morphology', 'location', 'symptom', 'history'}
propositions['category'] = propositions['category'].astype(str).str.casefold()
unknown_categories = set(propositions['category']) - required_categories
if unknown_categories:
    raise ValueError(f'Unexpected observable proposition categories: {sorted(unknown_categories)}')

# This notebook writes derived review files only; labels.npz is never opened for writing.
SURVEY_DIR.mkdir(parents=True, exist_ok=True)
summary_lines = []

def add_summary(section, frame):
    summary_lines.append(f'## {section}')
    summary_lines.append(frame.to_csv(index=False).strip())
    summary_lines.append('')

def label_counts(values):
    values = np.asarray(values)
    total = int(values.size)
    n_present = int(np.count_nonzero(values == 1))
    n_absent = int(np.count_nonzero(values == 0))
    n_unknown = int(np.count_nonzero(values == -1))
    return {
        'n_total': total,
        'n_present': n_present,
        'n_absent': n_absent,
        'n_unknown': n_unknown,
        'rate_present': n_present / total if total else np.nan,
        'rate_absent': n_absent / total if total else np.nan,
        'rate_unknown': n_unknown / total if total else np.nan,
    }

N_CASES, N_PROPOSITIONS = arrays['label_final'].shape
print(f'Read-only inputs validated: {N_CASES:,} cases × {N_PROPOSITIONS:,} propositions')

## 1. Thống kê số lượng và tỷ lệ nhãn

Thống kê nhãn cuối cùng theo split và nhóm mệnh đề; thống kê riêng nguồn văn bản và ảnh. Các ô ảnh chỉ tính khi `p_img` hữu hạn.

In [ ]:
splits = arrays['split'].astype(str)
label_final = arrays['label_final']
label_text = arrays['label_text']
label_img = arrays['label_img']

split_rows = []
category_rows = []
source_rows = []
for split_name in ('ALL', 'train', 'val', 'test'):
    split_mask = np.ones(N_CASES, dtype=bool) if split_name == 'ALL' else splits == split_name
    split_rows.append({'split': split_name, **label_counts(label_final[split_mask, :].ravel())})
    for category in sorted(required_categories):
        column_mask = propositions['category'].eq(category).to_numpy()
        category_rows.append({
            'split': split_name, 'category': category,
            **label_counts(label_final[np.ix_(split_mask, column_mask)].ravel()),
        })
    text_values = label_text[split_mask, :].ravel()
    image_values = label_img[split_mask, :]
    valid_image = np.isfinite(arrays['p_img'][split_mask, :])
    source_rows.append({'split': split_name, 'source': 'text', **label_counts(text_values)})
    source_rows.append({'split': split_name, 'source': 'image', **label_counts(image_values[valid_image])})

label_counts_by_split = pd.DataFrame(split_rows)
label_counts_by_category = pd.DataFrame(category_rows)
label_counts_by_source = pd.DataFrame(source_rows)
label_counts_by_split.to_csv(SURVEY_DIR / 'label_counts_by_split.csv', index=False)
label_counts_by_category.to_csv(SURVEY_DIR / 'label_counts_by_category.csv', index=False)
label_counts_by_source.to_csv(SURVEY_DIR / 'label_counts_by_source.csv', index=False)

train_mask = splits == 'train'
stat_rows = []
ratio_rows = []
for column, proposition in propositions.iterrows():
    final_train = label_final[train_mask, column]
    train_counts = label_counts(final_train)
    all_counts = label_counts(label_final[:, column])
    train_ratio = train_counts['n_present'] / train_counts['n_absent'] if train_counts['n_absent'] else np.inf
    stat_rows.append({
        'proposition_id': proposition['proposition_id'],
        'category': proposition['category'],
        'n_present': all_counts['n_present'],
        'n_absent': all_counts['n_absent'],
        'n_unknown': all_counts['n_unknown'],
        'n_train_labels': train_counts['n_present'] + train_counts['n_absent'],
        'train_positive_to_negative_ratio': train_ratio,
    })
    ratio_rows.append({
        'proposition_id': proposition['proposition_id'],
        'n_present_train': train_counts['n_present'],
        'n_absent_train': train_counts['n_absent'],
        'positive_to_negative_ratio': train_ratio,
    })

stats_per_proposition = pd.DataFrame(stat_rows)
train_positive_negative_ratio = pd.DataFrame(ratio_rows)
stats_per_proposition.to_csv(SURVEY_DIR / 'stats_per_proposition.csv', index=False)
train_positive_negative_ratio.to_csv(SURVEY_DIR / 'train_positive_negative_ratio.csv', index=False)

add_summary('1. Label counts and rates', label_counts_by_split)
add_summary('1. Train positive-to-negative ratio', train_positive_negative_ratio[['n_present_train', 'n_absent_train', 'positive_to_negative_ratio']].agg(['sum', 'min', 'max']).reset_index())
display(label_counts_by_split)
display(label_counts_by_category)
display(label_counts_by_source)

## 2. Tác động của bước xác minh LLM

Chỉ tổng hợp các ô đã gửi của nhóm A, B và D; không gọi API.

In [ ]:
group_codes = {'A': 1, 'B': 2, 'D': 4}
llm_rows = []
for split_name in ('train', 'val', 'test', 'ALL'):
    split_mask = np.ones(N_CASES, dtype=bool) if split_name == 'ALL' else splits == split_name
    for group_name, group_code in group_codes.items():
        selected = (arrays['llm_group'] == group_code) & split_mask[:, None]
        before = arrays['label_text_nli']
        after = arrays['label_text']
        llm_rows.append({
            'split': split_name,
            'group': group_name,
            'sent_cells': int(selected.sum()),
            'changed_1_to_unknown': int((selected & (before == 1) & (after == -1)).sum()),
            'changed_1_to_0': int((selected & (before == 1) & (after == 0)).sum()),
            'changed_unknown_to_0': int((selected & (before == -1) & (after == 0)).sum()),
            'quote_invalid': int((selected & (arrays['llm_quote_invalid'] == 1)).sum()),
        })

llm_impact_by_split = pd.DataFrame(llm_rows)
llm_impact_summary = llm_impact_by_split.query("split == 'ALL'").drop(columns='split')
llm_impact_by_split.to_csv(SURVEY_DIR / 'llm_impact_by_split.csv', index=False)
llm_impact_summary.to_csv(SURVEY_DIR / 'llm_impact_summary.csv', index=False)
add_summary('2. LLM impact by group', llm_impact_by_split)
display(llm_impact_by_split)

## 3. Histogram các xác suất gán nhãn

Các histogram sử dụng cùng miền $[0,1]$ và 50 bins; bảng số đếm giúp review không cần mở hình.

In [ ]:
BINS = np.linspace(0.0, 1.0, 51)
histogram_spec = {
    'p_entail': (arrays['p_entail'], [0.9]),
    'p_contra': (arrays['p_contra'], [0.8]),
    'p_img': (arrays['p_img'], [0.2, 0.8]),
}
hist_rows = []
hist_summary = []
for name, (scores, thresholds) in histogram_spec.items():
    valid = np.asarray(scores, dtype=np.float32).ravel()
    valid = valid[np.isfinite(valid)]
    counts, edges = np.histogram(valid, bins=BINS, range=(0.0, 1.0))
    for left, right, count in zip(edges[:-1], edges[1:], counts):
        hist_rows.append({'score': name, 'bin_left': left, 'bin_right': right, 'count': int(count)})
    fig, axis = plt.subplots(figsize=(8, 4))
    axis.hist(valid, bins=BINS, range=(0.0, 1.0), color='#4C78A8', edgecolor='white')
    for threshold in thresholds:
        axis.axvline(threshold, color='#E45756', linestyle='--', linewidth=2)
    axis.set(xlim=(0.0, 1.0), xlabel=name, ylabel='count', title=f'{name} (n_valid={len(valid):,})')
    fig.tight_layout()
    fig.savefig(SURVEY_DIR / f'{name}_histogram.png', dpi=160)
    plt.show()
    plt.close(fig)
    hist_summary.append({'score': name, 'n_valid': int(len(valid))})

probability_histogram_counts = pd.DataFrame(hist_rows)
probability_histogram_counts.to_csv(SURVEY_DIR / 'probability_histogram_counts.csv', index=False)
add_summary('3. Valid probability observations', pd.DataFrame(hist_summary))

## 4. Mô phỏng ngưỡng `min_labels`

Mô phỏng chỉ dùng nhãn cuối cùng của split train. Mỗi bệnh có ít nhất một mệnh đề không đạt ngưỡng sẽ được tính là cần fallback.

In [ ]:
train_determined = (label_final[train_mask, :] != -1).sum(axis=0)
propositions['n_train_labels'] = train_determined
simulation_rows = []
by_disease_rows = []
for min_labels in (1, 3, 5, 10):
    eligible = propositions['n_train_labels'].ge(min_labels)
    per_disease = propositions.assign(eligible=eligible).groupby('disease_id', as_index=False).agg(
        n_propositions=('proposition_id', 'size'),
        n_eligible=('eligible', 'sum'),
    )
    per_disease['min_labels'] = min_labels
    per_disease['requires_fallback'] = per_disease['n_eligible'].eq(0)
    by_disease_rows.extend(per_disease.to_dict('records'))
    simulation_rows.append({
        'min_labels': min_labels,
        'n_retained_propositions': int(eligible.sum()),
        'min_retained_propositions_per_disease': int(per_disease['n_eligible'].min()),
        'n_diseases_using_fallback': int(per_disease['requires_fallback'].sum()),
    })

min_labels_simulation = pd.DataFrame(simulation_rows)
min_labels_by_disease = pd.DataFrame(by_disease_rows)
min_labels_simulation.to_csv(SURVEY_DIR / 'min_labels_simulation.csv', index=False)
min_labels_by_disease.to_csv(SURVEY_DIR / 'min_labels_by_disease.csv', index=False)
add_summary('4. min_labels simulation', min_labels_simulation)
display(min_labels_simulation)

## 5. Kiểm tra rò rỉ nhãn với `own_share`

`own_share` dùng ánh xạ ca → bệnh duy nhất đã được lưu trong `meta.json`; notebook dừng rõ ràng nếu metadata này không tồn tại hoặc không khớp với `case_id`.

In [ ]:
case_disease_rows = meta.get('case_disease')
if not isinstance(case_disease_rows, list):
    raise ValueError(
        'meta.json must contain case_disease: a list of {case_id, disease_id} rows to compute own_share '
        'without inferring a case disease from weak labels.'
    )
case_disease = pd.DataFrame(case_disease_rows)
if not {'case_id', 'disease_id'} <= set(case_disease.columns):
    raise ValueError('meta.json case_disease must contain case_id and disease_id.')
if case_disease['case_id'].astype(str).duplicated().any():
    raise ValueError('meta.json case_disease has duplicate case_id values.')
case_disease_map = dict(zip(case_disease['case_id'].astype(str), case_disease['disease_id'].astype(str)))
case_ids = arrays['case_id'].astype(str)
missing_case_ids = sorted(set(case_ids) - set(case_disease_map))
if missing_case_ids:
    raise ValueError(f'case_disease misses {len(missing_case_ids)} case_id values; e.g. {missing_case_ids[:5]}')
case_diseases = np.asarray([case_disease_map[case_id] for case_id in case_ids])

own_rows = []
for column, proposition in propositions.iterrows():
    own_cases = case_diseases == str(proposition['disease_id'])
    before_positive = arrays['label_text_nli'][:, column] == 1
    after_positive = arrays['label_text'][:, column] == 1
    n_before = int(before_positive.sum())
    n_after = int(after_positive.sum())
    own_rows.append({
        'scope': 'all_splits',
        'proposition_id': proposition['proposition_id'],
        'category': proposition['category'],
        'disease_id': proposition['disease_id'],
        'n_positive_before_llm': n_before,
        'n_positive_after_llm': n_after,
        'own_share_before_llm': int((before_positive & own_cases).sum()) / n_before if n_before else np.nan,
        'own_share_after_llm': int((after_positive & own_cases).sum()) / n_after if n_after else np.nan,
    })

own_share_per_proposition = pd.DataFrame(own_rows)
own_share_ge_0_9 = own_share_per_proposition.loc[
    own_share_per_proposition['own_share_before_llm'].ge(0.9)
    | own_share_per_proposition['own_share_after_llm'].ge(0.9)
].copy()
own_share_per_proposition.to_csv(SURVEY_DIR / 'own_share_per_proposition.csv', index=False)
own_share_ge_0_9.to_csv(SURVEY_DIR / 'own_share_ge_0_9.csv', index=False)
own_summary = pd.DataFrame([{
    'n_propositions_own_share_before_llm_ge_0_9': int(own_share_per_proposition['own_share_before_llm'].ge(0.9).sum()),
    'n_propositions_own_share_after_llm_ge_0_9': int(own_share_per_proposition['own_share_after_llm'].ge(0.9).sum()),
}])
add_summary('5. own_share >= 0.9', own_summary)
display(own_share_ge_0_9)

# Numeric tables only: no interpretation is written to the Markdown summary.
(SURVEY_DIR / 'survey_summary.md').write_text('\n'.join(summary_lines) + '\n', encoding='utf-8')
print(f'Wrote review artifacts to: {SURVEY_DIR}')